In [1]:
import torch



A = torch.randn(size=(3, 10, 5))


In [5]:
U,S,V = torch.pca_lowrank(A, q=3, center=False, niter=2)

print('U shape: ', U.shape)
print('S shape: ', S.shape)
print('V shape: ', V.shape)

SV = torch.einsum('bq,bdq->bqd', S, V)
A_reconstructed = (U @ SV)
rec_loss = ((A - A_reconstructed)**2).sum()
print('Reconstruction loss: ', rec_loss)

U shape:  torch.Size([3, 10, 3])
S shape:  torch.Size([3, 3])
V shape:  torch.Size([3, 5, 3])
Reconstruction loss:  tensor(21.6775)


tensor([[[ 0.4462, -0.3831,  0.4726],
         [-0.1922,  0.7493,  1.3575],
         [-1.2945,  0.6471, -0.2770],
         [-1.4290,  0.1846, -1.8250],
         [-0.9136,  1.7711, -0.2996],
         [ 2.6010,  1.7931, -0.9371],
         [ 0.2491,  1.6132,  1.2424],
         [-1.9282,  1.0323, -0.1093],
         [ 0.2594,  0.0425, -0.3392],
         [-2.1037,  0.0247,  0.5632]],

        [[-1.6955, -1.4876,  1.4380],
         [-0.7335, -0.6014, -0.1515],
         [ 0.2819, -0.6520,  0.1093],
         [ 2.5700, -0.0511,  1.3244],
         [-0.3282,  0.2826,  1.1484],
         [ 0.8800,  0.4522,  0.1547],
         [ 0.7054,  1.2741,  0.6909],
         [-0.2474,  0.0731, -0.6152],
         [-2.4791,  1.7508,  0.5932],
         [-0.4712, -1.0743,  0.0759]],

        [[-2.5716,  0.3619,  0.0758],
         [ 1.6462,  0.2430, -0.1602],
         [-1.4511, -1.3727, -1.2756],
         [-0.0889, -1.3663,  0.3303],
         [-1.2380,  1.2756,  0.1723],
         [ 0.7655,  0.5313, -0.5675],
        

In [97]:
U,S,V = torch.pca_lowrank(A, q=5, center=True, niter=2)

print('U shape: ', U.shape)
print('S shape: ', S.shape)
print('V shape: ', V.shape)

SV = torch.einsum('bq,bdq->bqd', S, V)
A_reconstructed = (U @ SV)
rec_loss = ((A - A_reconstructed)**2).sum()
print('Reconstruction loss: ', rec_loss)

U shape:  torch.Size([3, 10, 5])
S shape:  torch.Size([3, 5])
V shape:  torch.Size([3, 5, 5])
Reconstruction loss:  tensor(14.6313)


In [125]:
U,S,V = torch.pca_lowrank(A, q=5, center=True, niter=2)

print('U shape: ', U.shape)
print('S shape: ', S.shape)
print('V shape: ', V.shape)

SV = torch.einsum('bq,bdq->bqd', S, V)
A_reconstructed = (U @ SV)
rec_loss = ((A - A_reconstructed)**2).sum()
print('Reconstruction loss: ', rec_loss)

centered_data = A - A.mean(dim=1).unsqueeze(1)
rec_loss = ((centered_data - A_reconstructed)**2).sum()
print('Reconstruction loss of centered data: ', rec_loss)


centered_data = A - A.mean(dim=1).unsqueeze(1)
rec_loss = ((A - (A_reconstructed + A.mean(dim=1).unsqueeze(1)))**2).sum()
print('Reconstruction loss of centered data: ', rec_loss)

U shape:  torch.Size([10, 10, 5])
S shape:  torch.Size([10, 5])
V shape:  torch.Size([10, 5, 5])
Reconstruction loss:  tensor(50.0729)
Reconstruction loss of centered data:  tensor(8.5319e-11)
Reconstruction loss of centered data:  tensor(8.5707e-11)


In [40]:
import torch

class PCAReconstructor:
    def __init__(self, q=5, niter=2):
        self.q = q
        self.niter = niter
        self.center = True
        

    def decompose(self, A):
        if self.center:
            mean = A.mean(dim=1, keepdim=True)
            
        U, S, V = torch.pca_lowrank(A, q=self.q, center=True, niter=self.niter)
        return U, S, V, mean

    def reconstruct(self, U, S, V, mean):    
        SV = torch.einsum('bq,bdq->bqd', S, V)
        A_reconstructed = U @ SV

        if self.center and mean is not None:
            A_reconstructed += mean
        return A_reconstructed
    
    def project(self, A):
        U, S, V, A_mean = self.decompose(A)
        A_projected = torch.einsum('bnd,bdi->bni', (A - A_mean), V)
        return A_projected, V, A_mean
    
    def reconstruct_from_projection(self, A_projected, V, A_mean):
        A_reconstructed = torch.einsum('bni,bdi->bnd', A_projected, V) + A_mean
        return A_reconstructed

    def compute_reconstruction_loss(self, A):
        U, S, V, mean = self.decompose(A)
        A_reconstructed = self.reconstruct(U, S, V, mean)
       
        loss = ((A - A_reconstructed) ** 2).sum()
        return loss

# Example usage
A = torch.randn(10, 10, 5)  # Example tensor
pca_reconstructor = PCAReconstructor(q=5, niter=2)

# Perform decomposition
U, S, V, mean = pca_reconstructor.decompose(A)

# Reconstruct the matrix
A_reconstructed = pca_reconstructor.reconstruct(U, S, V, mean)

# Calculate reconstruction loss
loss = pca_reconstructor.compute_reconstruction_loss(A)
print('Reconstruction loss:', loss)

# Check the projection and reconstruction from projection
A_projected, V, A_mean = pca_reconstructor.project(A)
A_reconstructed_from_projection = pca_reconstructor.reconstruct_from_projection(A_projected, V, A_mean)
loss = ((A_reconstructed_from_projection - A_reconstructed)**2).sum()
print('Reconstruction loss project and back:', loss)



Reconstruction loss: tensor(3.1390e-10, device='cuda:0')
Reconstruction loss project and back: tensor(8.5508e-10, device='cuda:0')
